In [1]:
import pandas as pd 
import numpy as np


# 1. SIMULATION & PARAMETER SETUP

# Change the seed for the different data generation.
# seed used for the main analysis is 42.
# seeds used for the multiple data analysis are 123,25,400,30,1,222,89,901,563,1023.
np.random.seed(42)

start_date = '2026-06-01'
days_to_simulate = 90
total_minutes = days_to_simulate * 1440
base_speed = 60

print(f"Starting 90-day continuous simulation from {start_date}...")

# Create a continuous 90-day timeline 
timestamps = pd.date_range(start=start_date, periods=total_minutes, freq='min')

# Pre-allocate dictionaries for all sensors
vol = {sensor: {'dir_a': np.zeros(total_minutes), 'dir_b': np.zeros(total_minutes)} for sensor in range(1, 11)}

# Vectorized time logic for extreme speed
time_steps = np.arange(total_minutes) % 1440
is_weekend = timestamps.dayofweek >= 5

# Base Daily Profiles 
# Weekend Math
main_entry_weekend = 30 + 25 * np.sin(np.pi * (time_steps - 400) / 720)
side_entry_base_weekend = 8 + 10 * np.sin(np.pi * (time_steps - 400) / 720)
prob_stay_weekend = np.full(total_minutes, 0.70)

# Weekday Math
main_entry_weekday = 40 + 25 * np.sin(np.pi * (time_steps - 300) / 360) + 35 * np.sin(np.pi * (time_steps - 840) / 360)
side_entry_base_weekday = 10 + 20 * np.sin(np.pi * (time_steps - 300) / 360)
prob_stay_weekday = 0.85 - 0.15 * np.sin(np.pi * (time_steps - 840) / 360)

# Merge profiles conditionally based on weekend/weekday
main_entry = np.where(is_weekend, main_entry_weekend, main_entry_weekday)
side_entry_base = np.where(is_weekend, side_entry_base_weekend, side_entry_base_weekday)

prob_stay = np.where(is_weekend, prob_stay_weekend, prob_stay_weekday)
prob_stay = np.clip(prob_stay, 0.50, 0.95)
prob_leave_side = (1 - prob_stay) / 2

# Probabilities
prob_side_merge_fwd = 0.40  
prob_side_merge_rev = 0.40  
prob_side_cross = 0.20      

# Generate Base Injections with Noise
vol[1]['dir_a'] = np.clip(main_entry + np.random.normal(0, 5, total_minutes), 10, 200)
vol[10]['dir_b'] = np.clip(main_entry + np.random.normal(0, 5, total_minutes), 10, 200)

for s in [2, 3, 5, 6, 8, 9]:
    vol[s]['dir_a'] = np.clip(side_entry_base + np.random.normal(0, 3, total_minutes), 2, 60)

# Helper function for true chronological shifting 
def timeline_shift(arr, mins, fill=30):
    return pd.Series(arr).shift(periods=mins, fill_value=fill).values

lag_mins = 5


# 2. CONTINUOUS NETWORK PROPAGATION


# FORWARD PROPAGATION (DIR_A) 
# Junction 1 (Sensors 1, 2, 3 -> Sensor 4)
main_in_a = vol[1]['dir_a']
vol[4]['dir_a'] = timeline_shift((main_in_a * prob_stay) + (vol[2]['dir_a'] * prob_side_merge_fwd) + (vol[3]['dir_a'] * prob_side_merge_fwd), lag_mins)
vol[2]['dir_b'] += (main_in_a * prob_leave_side) + (vol[3]['dir_a'] * prob_side_cross) 
vol[3]['dir_b'] += (main_in_a * prob_leave_side) + (vol[2]['dir_a'] * prob_side_cross)

# Junction 2 (Sensors 4, 5, 6 -> Sensor 7)
main_in_a = vol[4]['dir_a']
vol[7]['dir_a'] = timeline_shift((main_in_a * prob_stay) + (vol[5]['dir_a'] * prob_side_merge_fwd) + (vol[6]['dir_a'] * prob_side_merge_fwd), lag_mins)
vol[5]['dir_b'] += (main_in_a * prob_leave_side) + (vol[6]['dir_a'] * prob_side_cross)
vol[6]['dir_b'] += (main_in_a * prob_leave_side) + (vol[5]['dir_a'] * prob_side_cross)

# Junction 3 (Sensors 7, 8, 9 -> Sensor 10)
main_in_a = vol[7]['dir_a']
vol[10]['dir_a'] = timeline_shift((main_in_a * prob_stay) + (vol[8]['dir_a'] * prob_side_merge_fwd) + (vol[9]['dir_a'] * prob_side_merge_fwd), lag_mins)
vol[8]['dir_b'] += (main_in_a * prob_leave_side) + (vol[9]['dir_a'] * prob_side_cross)
vol[9]['dir_b'] += (main_in_a * prob_leave_side) + (vol[8]['dir_a'] * prob_side_cross)


#  REVERSE PROPAGATION (DIR_B)
# Junction 3 Reverse (Sensors 10, 8, 9 -> Sensor 7)
main_in_b = vol[10]['dir_b']
vol[7]['dir_b'] = timeline_shift((main_in_b * prob_stay) + (vol[8]['dir_a'] * prob_side_merge_rev) + (vol[9]['dir_a'] * prob_side_merge_rev), lag_mins)
vol[8]['dir_b'] += (main_in_b * prob_leave_side) 
vol[9]['dir_b'] += (main_in_b * prob_leave_side)

# Junction 2 Reverse (Sensors 7, 5, 6 -> Sensor 4)
main_in_b = vol[7]['dir_b']
vol[4]['dir_b'] = timeline_shift((main_in_b * prob_stay) + (vol[5]['dir_a'] * prob_side_merge_rev) + (vol[6]['dir_a'] * prob_side_merge_rev), lag_mins)
vol[5]['dir_b'] += (main_in_b * prob_leave_side)
vol[6]['dir_b'] += (main_in_b * prob_leave_side)

# Junction 1 Reverse (Sensors 4, 2, 3 -> Sensor 1)
main_in_b = vol[4]['dir_b']
vol[1]['dir_b'] = timeline_shift((main_in_b * prob_stay) + (vol[2]['dir_a'] * prob_side_merge_rev) + (vol[3]['dir_a'] * prob_side_merge_rev), lag_mins)
vol[2]['dir_b'] += (main_in_b * prob_leave_side)
vol[3]['dir_b'] += (main_in_b * prob_leave_side)


# 3. BUILD THE DATAFRAME

print("Compiling DataFrame...")
dfs_to_concat = []

for sensor_id in range(1, 11):
    for direction in ['dir_a', 'dir_b']:
        
        final_volumes = np.clip(np.round(vol[sensor_id][direction]), 0, None)
        speed_noise = np.random.normal(0, 2.5, total_minutes)
        average_speeds = np.clip(base_speed - (final_volumes * 0.12) + speed_noise, 10, 70)
        
        temp_df = pd.DataFrame({
            'Timestamp': timestamps,
            'Sensor_ID': sensor_id,
            'Direction': direction,
            'Vehicle_Count': final_volumes.astype(int),
            'Average_Speed_mph': np.round(average_speeds, 1)
        })
        dfs_to_concat.append(temp_df)

df = pd.concat(dfs_to_concat, ignore_index=True)
df['Day_Name'] = df['Timestamp'].dt.day_name()
df = df.sort_values(by=['Timestamp', 'Sensor_ID', 'Direction']).reset_index(drop=True)


# 4. THE GRADUAL SHOCKWAVE FUNCTION

def apply_gradual_shockwave(df, sensor_id, direction, start_time, end_time, 
                            target_speed=None, vol_multiplier=0.5, 
                            ramp_down_mins=15, recovery_mins=25, effect_type='gridlock'):
    mask = (df['Sensor_ID'] == sensor_id) & (df['Direction'] == direction) & \
           (df['Timestamp'] >= start_time) & (df['Timestamp'] <= end_time)
    
    idx = df[mask].index
    if len(idx) == 0:
        return
        
    t_start = pd.to_datetime(start_time)
    t_end = pd.to_datetime(end_time)
    
    elapsed_mins = (df.loc[idx, 'Timestamp'] - t_start).dt.total_seconds() / 60.0
    remaining_mins = (t_end - df.loc[idx, 'Timestamp']).dt.total_seconds() / 60.0
    
    severity = np.ones(len(idx))
    
    ramp_down_mask = elapsed_mins < ramp_down_mins
    severity[ramp_down_mask] = elapsed_mins[ramp_down_mask] / ramp_down_mins
    
    recovery_mask = remaining_mins < recovery_mins
    severity[recovery_mask] = remaining_mins[recovery_mask] / recovery_mins
    
    if effect_type == 'gridlock' and target_speed is not None:
        jam_speeds = np.random.normal(target_speed, 3, len(idx))
        df.loc[idx, 'Average_Speed_mph'] = (
            df.loc[idx, 'Average_Speed_mph'] * (1 - severity) + jam_speeds * severity
        ).round(1)
        
    target_volumes = df.loc[idx, 'Vehicle_Count'] * vol_multiplier
    df.loc[idx, 'Vehicle_Count'] = (
        df.loc[idx, 'Vehicle_Count'] * (1 - severity) + target_volumes * severity
    ).round().astype(int)


# 5. EXECUTE NETWORK-WIDE SHOCKWAVES

incidents = [
    {"start": "2026-06-23 16:30:00", "end": "2026-06-23 18:45:00"},  # Tuesday PM
    {"start": "2026-07-09 07:15:00", "end": "2026-07-09 09:30:00"},  # Thursday AM
    {"start": "2026-07-25 11:00:00", "end": "2026-07-25 13:15:00"}   # Saturday Midday
]

for num, jam in enumerate(incidents, start=1):
    jam_start = pd.to_datetime(jam["start"])
    jam_end = pd.to_datetime(jam["end"])
    
    print(f"Injecting Incident #{num} starting on {jam_start.day_name()} at {jam_start.strftime('%H:%M')}...")

    # A. DIRECTION A IMPACTS 
    apply_gradual_shockwave(df, 4, 'dir_a', jam_start, jam_end, target_speed=12, vol_multiplier=0.4, ramp_down_mins=15, recovery_mins=25, effect_type='gridlock')
    apply_gradual_shockwave(df, 2, 'dir_a', jam_start + pd.Timedelta(minutes=4), jam_end + pd.Timedelta(minutes=10), target_speed=11, vol_multiplier=0.4, ramp_down_mins=10, recovery_mins=30, effect_type='gridlock')
    apply_gradual_shockwave(df, 3, 'dir_a', jam_start + pd.Timedelta(minutes=4), jam_end + pd.Timedelta(minutes=10), target_speed=11, vol_multiplier=0.4, ramp_down_mins=10, recovery_mins=30, effect_type='gridlock')
    apply_gradual_shockwave(df, 1, 'dir_a', jam_start + pd.Timedelta(minutes=12), jam_end + pd.Timedelta(minutes=20), target_speed=15, vol_multiplier=0.5, ramp_down_mins=15, recovery_mins=30, effect_type='gridlock')
    apply_gradual_shockwave(df, 7, 'dir_a', jam_start + pd.Timedelta(minutes=3), jam_end, vol_multiplier=0.5, ramp_down_mins=15, recovery_mins=20, effect_type='starvation')
    apply_gradual_shockwave(df, 10, 'dir_a', jam_start + pd.Timedelta(minutes=7), jam_end, vol_multiplier=0.6, ramp_down_mins=15, recovery_mins=20, effect_type='starvation')
    
    for side_sensor in [5, 6, 8, 9]:
        apply_gradual_shockwave(df, side_sensor, 'dir_a', jam_start + pd.Timedelta(minutes=10), jam_end, vol_multiplier=0.70, ramp_down_mins=20, recovery_mins=20, effect_type='starvation')

    # B. DIRECTION B IMPACTS 
    for s in [4, 5, 6]:
        apply_gradual_shockwave(df, s, 'dir_b', jam_start + pd.Timedelta(minutes=5), jam_end, vol_multiplier=0.65, ramp_down_mins=15, recovery_mins=20, effect_type='starvation')
    for s in [1, 2, 3]:
        apply_gradual_shockwave(df, s, 'dir_b', jam_start + pd.Timedelta(minutes=10), jam_end + pd.Timedelta(minutes=15), vol_multiplier=0.60, ramp_down_mins=15, recovery_mins=25, effect_type='starvation')


# 6. FINAL CLEANUP 

# Reorder columns to match original structure exactly
df = df[['Timestamp', 'Day_Name', 'Sensor_ID', 'Direction', 'Vehicle_Count', 'Average_Speed_mph']]

# Ensure strings for export
df['Timestamp_str'] = df['Timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

print("All 3 gradual, topology-aware incidents injected successfully!")

Starting 90-day continuous simulation from 2026-06-01...
Compiling DataFrame...
Injecting Incident #1 starting on Tuesday at 16:30...
Injecting Incident #2 starting on Thursday at 07:15...
Injecting Incident #3 starting on Saturday at 11:00...
All 3 gradual, topology-aware incidents injected successfully!


In [2]:
# 7. RESHAPING FOR MICE (Wide Format) 
print("Pivoting data to wide format for MICE...")

# Pivot the dataset
# Index: The exact time (keeps rows distinct per minute)
# Columns: We split by Sensor and Direction
# Values: The metrics we want to track
df_wide = df.pivot_table(
    index=['Timestamp', 'Day_Name'], 
    columns=['Sensor_ID', 'Direction'], 
    values=['Vehicle_Count', 'Average_Speed_mph']
)

# Flatten the MultiIndex Columns

df_wide.columns = [
    f"{'Vol' if metric == 'Vehicle_Count' else 'Spd'}_S{sensor}_{direction}" 
    for metric, sensor, direction in df_wide.columns
]

# Reset the index so Timestamp and day_name return to being normal columns
df_wide = df_wide.reset_index()

# Extract numeric time features for the MICE algorithm
# MICE cannot perform math on Datetime strings, so we give it numeric columns
df_wide['Timestamp'] = pd.to_datetime(df_wide['Timestamp'])
df_wide['Hour'] = df_wide['Timestamp'].dt.hour
df_wide['Minute'] = df_wide['Timestamp'].dt.minute
df_wide['DayOfWeek'] = df_wide['Timestamp'].dt.dayofweek # 0=Monday, 6=Sunday

# Reorder columns to put time features up front
cols = ['Timestamp', 'Day_Name', 'DayOfWeek', 'Hour', 'Minute'] + [c for c in df_wide.columns if c.startswith(('Vol', 'Spd'))]
df_wide = df_wide[cols]

print(f"Wide Format Complete. Total Rows: {len(df_wide)}, Total Columns: {len(df_wide.columns)}")
print("-" * 50)
print(df_wide.head())




Pivoting data to wide format for MICE...
Wide Format Complete. Total Rows: 129600, Total Columns: 45
--------------------------------------------------
            Timestamp Day_Name  DayOfWeek  Hour  Minute  Spd_S1_dir_a  \
0 2026-06-01 00:00:00   Monday          0     0       0          59.9   
1 2026-06-01 00:01:00   Monday          0     0       1          59.5   
2 2026-06-01 00:02:00   Monday          0     0       2          59.3   
3 2026-06-01 00:03:00   Monday          0     0       3          59.6   
4 2026-06-01 00:04:00   Monday          0     0       4          56.2   

   Spd_S1_dir_b  Spd_S2_dir_a  Spd_S2_dir_b  Spd_S3_dir_a  ...  Vol_S6_dir_a  \
0          54.4          59.9          64.5          63.8  ...           2.0   
1          58.3          57.8          57.0          64.3  ...           2.0   
2          56.6          60.0          57.1          53.7  ...           2.0   
3          54.0          62.0          52.5          59.1  ...           2.0   
4        

In [3]:
# Export the MICE-ready dataset
# change the csv file name for each of the data generated
df_wide.to_csv('MICE_wide_3jamnew_10.csv', index=False)